# Checkpoint 2 — Surrogate FMM (sklearn)

Strategia: addestriamo un **MLP surrogato** (sklearn) che rimpiazza la chiamata lenta a `anisotropic_FMM_GPT` dentro l'ottimizzatore.

- **Input surrogate (6 scalari):** `(x_src, y_src, xc, yc, r, a_ratio_sqrt)`
- **Output surrogate (20 scalari):** tempi di attivazione normalizzati ai 20 sensori
- **Dataset generazione:** joblib parallelizza le chiamate FMM
- **Risparmio:** FMM ~1-10s/call → surrogate <1ms/call
- **Pipeline:** identica alla TerzaVersione_parallela

In [1]:
import numpy as np
import heapq
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

## FMM originale (usata SOLO per generare il training set del surrogate)

In [2]:
def anisotropic_FMM_GPT(
    x0, y0, t0, X, Y, speed_input, sigma_11, sigma_12, sigma_22, N_edge=None
):
    nrows, ncols = X.shape
    if N_edge is None:
        N_edge = nrows - 1
    INF = 1e4
    P = np.linspace(0.0, 1.0, 11)
    j0 = np.abs(X[0, :] - x0).argmin()
    i0 = np.abs(Y[:, 0] - y0).argmin()
    T = np.full((nrows, ncols), INF, dtype=float)
    T_final = np.full((nrows, ncols), INF, dtype=float)
    T[i0, j0] = t0
    T_final[i0, j0] = t0
    known = np.zeros((nrows, ncols), dtype=bool)
    heap = [(t0, i0, j0)]

    def get_speed(i, j, x, y):
        if callable(speed_input): return speed_input(x, y)
        elif np.isscalar(speed_input): return speed_input
        else: return speed_input[i, j]

    def metric_dist(xa, ya, xb, yb, sv):
        dx = xa - xb; dy = ya - yb
        return np.sqrt(sigma_11*dx*dx + 2.*sigma_12*dx*dy + sigma_22*dy*dy) / sv

    def update_fun(T_xi, T_x, T_y, X_xi, X_x, X_y, Y_xi, Y_x, Y_y, flag_known, i_xi, j_xi):
        sv = get_speed(i_xi, j_xi, X_xi, Y_xi)
        if flag_known:
            dx = P*(X_xi-X_x) + (1.-P)*(X_xi-X_y)
            dy = P*(Y_xi-Y_x) + (1.-P)*(Y_xi-Y_y)
            cand = (P*T_x + (1.-P)*T_y + np.sqrt(sigma_11*dx*dx + 2.*sigma_12*dx*dy + sigma_22*dy*dy)/sv).min()
        else:
            cand = T_x + metric_dist(X_xi, Y_xi, X_x, Y_x, sv)
        return min(T_xi, cand)

    side_map = {
        (-1, 0): [(-1,+1),(-1,-1)], (+1, 0): [(+1,+1),(+1,-1)],
        (0, -1): [(+1,-1),(-1,-1)], (0, +1): [(+1,+1),(-1,+1)],
    }
    while heap:
        t_sel, i_sel, j_sel = heapq.heappop(heap)
        if known[i_sel, j_sel] or t_sel > T_final[i_sel, j_sel]:
            continue
        known[i_sel, j_sel] = True
        for di, dj in [(-1,0),(1,0),(0,-1),(0,+1)]:
            ni, nj = i_sel+di, j_sel+dj
            if not (0<=ni<nrows and 0<=nj<ncols) or known[ni, nj]:
                continue
            new_T = INF
            for sdi, sdj in side_map[(di, dj)]:
                si, sj = i_sel+sdi, j_sel+sdj
                if not (0<=si<nrows and 0<=sj<ncols):
                    continue
                new_T = update_fun(
                    new_T, T_final[i_sel,j_sel], T_final[si,sj],
                    X[ni,nj], X[i_sel,j_sel], X[si,sj],
                    Y[ni,nj], Y[i_sel,j_sel], Y[si,sj],
                    known[si,sj], ni, nj
                )
            if new_T < T_final[ni, nj]:
                T[ni, nj] = new_T; T_final[ni, nj] = new_T
                heapq.heappush(heap, (new_T, ni, nj))
    return T_final

## Setup griglia e sensori

In [3]:
X_mesh_vel, Y_mesh_vel = np.meshgrid(np.linspace(-2.0, 2.0, 201), np.linspace(-2.0, 2.0, 201))

Spline_spacing = [0.5, 0.7, 1.3, 1.5]
Angles_spacing = [
    np.pi/10.0+0, np.pi/10.0+(2)*np.pi/5, np.pi/10.0+(4)*np.pi/5,
    np.pi/10.0+(6)*np.pi/5, np.pi/10.0+(8)*np.pi/5
]

meas_indices_vel = []
for j in range(5):
    for i in range(4):
        MeasMat = (X_mesh_vel - Spline_spacing[i]*np.cos(Angles_spacing[j]))**2 \
                + (Y_mesh_vel - Spline_spacing[i]*np.sin(Angles_spacing[j]))**2
        i_sel, j_sel = np.unravel_index(MeasMat.argmin(), MeasMat.shape)
        meas_indices_vel.append((i_sel, j_sel))

idx_vel = np.array(meas_indices_vel)  # (20, 2)
print(f'Sensor indices shape: {idx_vel.shape}')

Sensor indices shape: (20, 2)


## Caricamento dati

In [4]:
signals           = np.load('checkpoint2dataset.npy')
impulse_sites     = np.load('checkpoint2FocalPoints.npy')
starting_times    = np.load('checkpoint2starting.npy')
a_ratios_training = np.load('checkpoint2anisotropy.npy')

reference_targets = [
    [-0.407,  0.299, 0.578],
    [ 0.81,   0.113, 0.394],
    [-0.816, -0.686, 0.274],
    [ 0.197, -0.242, 0.042],
    [ 0.647, -0.69,  0.244]
]

EST_V_BASE = 67.96
EST_V_SLOW = 19.80

print('signals:', signals.shape)
print('impulse_sites:', impulse_sites.shape)
print('starting_times:', starting_times.shape)

signals: (5, 20, 501)
impulse_sites: (5, 2, 2)
starting_times: (5, 2)


## Helper: FMM → 20 tempi sensori normalizzati

Questa è la funzione che il surrogate impara a replicare.

In [5]:
def run_fmm_get_sensor_times(x_src, y_src, xc, yc, r, a_ratio_sqrt,
                              v_base=None, v_slow=None, k=50.0):
    """
    Chiama FMM e restituisce i 20 tempi normalizzati ai sensori.
    t_src=0 fisso: la normalizzazione min=0 rende t_src irrilevante.
    a_ratio_sqrt = sqrt(a_ratio) come nel codice originale.
    """
    if v_base is None: v_base = EST_V_BASE
    if v_slow is None: v_slow = EST_V_SLOW

    a_ratio = float(a_ratio_sqrt ** 2)
    sigma_11, sigma_12, sigma_22 = 1.0, 0.0, a_ratio

    dist = np.hypot(X_mesh_vel - xc, Y_mesh_vel - yc)
    speed_matrix = v_slow + 0.5*(1 + np.tanh(k*(dist - r)))*(v_base - v_slow)

    T_map = anisotropic_FMM_GPT(
        x_src, y_src, 0.0,
        X_mesh_vel, Y_mesh_vel, speed_matrix,
        sigma_11, sigma_12, sigma_22
    ) * 1000.0  # ms

    times = T_map[idx_vel[:, 0], idx_vel[:, 1]]
    return times - times.min()

## Generazione dataset con joblib

**Gira una sola volta**, poi salva su disco. Rilanciando il notebook salta direttamente al caricamento.

In [ ]:
# import os
# import time
# from joblib import Parallel, delayed

# SURROGATE_DATASET_FILE = 'surrogate_dataset.npy'

# # Aumenta N_SAMPLES per più accuratezza (5000 = ~10-15 min con parallelismo)
# N_SAMPLES = 5000

# def _generate_one(seed):
#     """
#     Worker joblib: campiona parametri casuali, chiama FMM, ritorna (features, labels).
#     Ogni worker è indipendente — nessuna variabile globale condivisa viene modificata.
#     """
#     import numpy as _np
#     import heapq as _hq

#     rng = _np.random.RandomState(seed)

#     x_src      = rng.uniform(-1.6,  1.6)
#     y_src      = rng.uniform(-1.6,  1.6)
#     xc         = rng.uniform(-1.5,  1.5)
#     yc         = rng.uniform(-1.5,  1.5)
#     r          = rng.uniform(0.02,  1.2)
#     a_ratio_sqrt = rng.uniform(1.0, 5.0)

#     try:
#         times = run_fmm_get_sensor_times(x_src, y_src, xc, yc, r, a_ratio_sqrt)
#         features = _np.array([x_src, y_src, xc, yc, r, a_ratio_sqrt], dtype=_np.float32)
#         return _np.concatenate([features, times.astype(_np.float32)])
#     except Exception:
#         return None


# if os.path.exists(SURROGATE_DATASET_FILE):
#     print(f'Dataset già presente ({SURROGATE_DATASET_FILE}) — skip generazione.')
#     data = np.load(SURROGATE_DATASET_FILE)
# else:
#     print(f'Generazione {N_SAMPLES} campioni FMM in parallelo (joblib)...')
#     t0 = time.time()

#     results = Parallel(n_jobs=-1, verbose=5, prefer='processes')(
#         delayed(_generate_one)(seed) for seed in range(N_SAMPLES)
#     )

#     results = [r for r in results if r is not None]
#     data = np.array(results, dtype=np.float32)
#     np.save(SURROGATE_DATASET_FILE, data)

#     elapsed = time.time() - t0
#     print(f'\nGenerati {len(results)} campioni in {elapsed/60:.1f} min')

# X_surr = data[:, :6]   # (N, 6)
# Y_surr = data[:, 6:]   # (N, 20)
# print(f'Dataset: X={X_surr.shape}, Y={Y_surr.shape}')

## Training del surrogate MLP (sklearn)

Usiamo `MLPRegressor` — nessun PyTorch richiesto.

In [6]:
import os
import time
from joblib import Parallel, delayed

SURROGATE_DATASET_FILE = 'chunk_15741.npy'

# Aumenta N_SAMPLES per più accuratezza (5000 = ~10-15 min con parallelismo)
N_SAMPLES = 15000

data = np.load(SURROGATE_DATASET_FILE)
X_surr = data[:, :6]   # (N, 6)
Y_surr = data[:, 6:]   # (N, 20)
print(f'Dataset: X={X_surr.shape}, Y={Y_surr.shape}')

Dataset: X=(15000, 6), Y=(15000, 20)


In [8]:
import joblib as jl
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

SURROGATE_MODEL_FILE  = 'surrogate_mlp.joblib'
SURROGATE_SCALER_FILE = 'surrogate_scaler.joblib'

if os.path.exists(SURROGATE_MODEL_FILE) and os.path.exists(SURROGATE_SCALER_FILE):
    print('Modello surrogate già addestrato — caricamento...')
    scaler_X = jl.load(SURROGATE_SCALER_FILE)
    surrogate = jl.load(SURROGATE_MODEL_FILE)
    print('Caricato!')
else:
    print('Training MLP surrogate (sklearn)...')
    t0 = time.time()

    X_train, X_val, Y_train, Y_val = train_test_split(
        X_surr, Y_surr, test_size=0.1, random_state=42
    )

    # Normalizzazione input
    scaler_X = StandardScaler()
    X_train_n = scaler_X.fit_transform(X_train)
    X_val_n   = scaler_X.transform(X_val)

    # MLP: 4 hidden layers, 256 neuroni ciascuno
    # Nota: sklearn MLPRegressor fa normalization interna opzionale,
    # ma la facciamo noi esternamente per controllo.
    surrogate = MLPRegressor(
        hidden_layer_sizes=(256, 256, 256, 256),
        activation='relu',
        solver='adam',
        alpha=1e-5,           # L2 regularization
        batch_size=256,
        learning_rate_init=1e-3,
        max_iter=300,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=20,
        random_state=42,
        verbose=True
    )

    surrogate.fit(X_train_n, Y_train)

    elapsed = time.time() - t0
    print(f'Training completato in {elapsed:.1f}s')

    # Salvataggio
    jl.dump(scaler_X,  SURROGATE_SCALER_FILE)
    jl.dump(surrogate, SURROGATE_MODEL_FILE)
    print(f'Modello salvato: {SURROGATE_MODEL_FILE}')

    # Validation MAE in ms
    Y_pred_val = surrogate.predict(X_val_n)
    mae_val = np.mean(np.abs(Y_pred_val - Y_val))
    print(f'Validation MAE: {mae_val:.3f} ms')

Modello surrogate già addestrato — caricamento...
Caricato!


## Validazione accuracy surrogate vs FMM vero

In [ ]:

# def surrogate_predict(x_src, y_src, xc, yc, r, a_ratio_sqrt):
#     """
#     Predice i 20 tempi ai sensori (ms, normalizzati min=0) con il surrogate.
#     Microsecondialmente veloce rispetto a FMM.
#     """
#     feat = np.array([[x_src, y_src, xc, yc, r, a_ratio_sqrt]], dtype=np.float32)
#     feat_n = scaler_X.transform(feat)
#     times = surrogate.predict(feat_n)[0]
#     times = times - times.min()
#     return times


# print('Validazione surrogate su 30 campioni test (FMM vero vs surrogate)...')
# np.random.seed(12345)
# maes = []
# for _ in range(30):
#     x_src = np.random.uniform(-1.5, 1.5)
#     y_src = np.random.uniform(-1.5, 1.5)
#     xc    = np.random.uniform(-1.5, 1.5)
#     yc    = np.random.uniform(-1.5, 1.5)
#     r     = np.random.uniform(0.05, 1.0)
#     a_sq  = np.random.uniform(1.0, 5.0)

#     true_t = run_fmm_get_sensor_times(x_src, y_src, xc, yc, r, a_sq)
#     pred_t = surrogate_predict(x_src, y_src, xc, yc, r, a_sq)
#     maes.append(np.mean(np.abs(true_t - pred_t)))

# print(f'MAE medio : {np.mean(maes):.3f} ms')
# print(f'MAE p90   : {np.percentile(maes, 90):.3f} ms')
# print(f'MAE max   : {np.max(maes):.3f} ms')

# if np.mean(maes) < 5.0:
#     print('\n✅ Surrogate OK — preciso abbastanza per rimpiazzare FMM nell\'ottimizzatore!')
# else:
#     print('\n⚠️  MAE > 5ms. Aumenta N_SAMPLES a 10000 e riaddestra.')

Validazione surrogate su 30 campioni test (FMM vero vs surrogate)...
MAE medio : 2.347 ms
MAE p90   : 4.685 ms
MAE max   : 8.298 ms

✅ Surrogate OK — preciso abbastanza per rimpiazzare FMM nell'ottimizzatore!


## Estrazione tempi misurati (Checkpoint 1)

In [10]:
from scipy.signal import argrelextrema, savgol_filter

def GroupSolutionCP1(signals_event):
    sig_filtered = savgol_filter(signals_event, window_length=11, polyorder=3, axis=1)
    n_sensors = sig_filtered.shape[0]
    timings = []
    min_dist = 100
    for i in range(n_sensors):
        ds = np.gradient(sig_filtered[i, :])
        min_indices = argrelextrema(ds, np.less)[0]
        selected = []
        if len(min_indices) > 0:
            candidates = min_indices[np.argsort(ds[min_indices])]
            for idx in candidates:
                if len(selected) == 0:
                    selected.append(idx)
                elif len(selected) == 1 and abs(idx - selected[0]) >= min_dist:
                    selected.append(idx)
                    break
            selected.sort()
        row = [int(selected[j]) if j < len(selected) else np.nan for j in range(2)]
        timings.append(row)
    return timings


# Pre-estrazione per tutti i pazienti
measured_times_beat1 = []
measured_times_beat2 = []

for idx_event in range(5):
    raw = GroupSolutionCP1(signals[idx_event, :, :])
    b1, b2 = [], []
    for s in range(5):
        for t in raw[s*4:(s+1)*4][::-1]:  # inverti ordine radiale
            b1.append(t[0])
            b2.append(t[1])
    measured_times_beat1.append(np.array(b1, dtype=float))
    measured_times_beat2.append(np.array(b2, dtype=float))

print('Tempi estratti per 5 scenari ✓')

Tempi estratti per 5 scenari ✓


## Calibrazione velocità (FMM vero, una sola volta)

In [ ]:
# from scipy.optimize import minimize
# from joblib import Parallel, delayed

# def _vel_error_single(idx, velocities):
#     v_base, v_slow = velocities
#     k = 50.0
#     xc, yc, r = reference_targets[idx]
#     a_ratio = float(a_ratios_training[idx].item()**2)
#     sigma_11, sigma_12, sigma_22 = 1.0, 0.0, a_ratio
#     dist = np.hypot(X_mesh_vel - xc, Y_mesh_vel - yc)
#     speed_grid = v_slow + 0.5*(1 + np.tanh(k*(dist - r)))*(v_base - v_slow)
#     total = 0.0
#     for beat_i, meas_t in [(0, measured_times_beat1[idx]), (1, measured_times_beat2[idx])]:
#         xi, yi = impulse_sites[idx][beat_i]
#         ti = starting_times[idx][beat_i] / 1000.0
#         sim = anisotropic_FMM_GPT(xi, yi, ti, X_mesh_vel, Y_mesh_vel, speed_grid,
#                                    sigma_11, sigma_12, sigma_22) * 1000.0
#         s = sim[idx_vel[:,0], idx_vel[:,1]]
#         s = s - np.nanmin(s)
#         m = meas_t - np.nanmin(meas_t)
#         mask = ~np.isnan(m)
#         if np.any(mask):
#             total += np.mean((s[mask] - m[mask])**2)
#     return total

# def objective_velocities(velocities):
#     errors = Parallel(n_jobs=5, backend='loky')(
#         delayed(_vel_error_single)(idx, velocities) for idx in range(5)
#     )
#     return sum(errors)

# print('Calibrazione velocità (usa FMM vero — pochi minuti)...')
# result_vel = minimize(
#     objective_velocities, [80.0, 20.0], method='Nelder-Mead',
#     bounds=[(40.0, 150.0), (5.0, 40.0)],
#     options={'xatol': 1.0, 'fatol': 0.1}
# )
# if result_vel.success:
#     EST_V_BASE, EST_V_SLOW = result_vel.x
#     print(f'✅ v_base={EST_V_BASE:.2f}, v_slow={EST_V_SLOW:.2f}')
# else:
#     EST_V_BASE, EST_V_SLOW = 67.96, 19.80
#     print(f'⚠️ Default: v_base={EST_V_BASE}, v_slow={EST_V_SLOW}')

Calibrazione velocità (usa FMM vero — pochi minuti)...
✅ v_base=67.96, v_slow=19.80


## Firma ufficiale Checkpoint 2

**Fase 1 (surrogate):** esplora ~1000+ configurazioni in secondi.  
**Fase 2 (FMM vero):** affina localmente con poche iterazioni dal minimo già trovato.

In [ ]:
from scipy.stats import pearsonr
from scipy.optimize import minimize
from joblib import Parallel, delayed

def surrogate_predict(x_src, y_src, xc, yc, r, a_ratio_sqrt):
    """
    Predice i 20 tempi ai sensori (ms, normalizzati min=0) con il surrogate.
    Microsecondialmente veloce rispetto a FMM.
    """
    feat = np.array([[x_src, y_src, xc, yc, r, a_ratio_sqrt]], dtype=np.float32)
    feat_n = scaler_X.transform(feat)
    times = surrogate.predict(feat_n)[0]
    times = times - times.min()
    return times


def GroupSolutionCP2(signals_event, x_0, y_0, t_0, x_1, y_1, t_1):
    """
    Stima centro e raggio della zona di conduzione lenta.
    Ritorna: [x_SC, y_SC, r_SC]
    """
    # --- 1. Estrazione tempi misurati ---
    raw = GroupSolutionCP1(signals_event)
    b1, b2 = [], []
    for s in range(5):
        for t in raw[s*4:(s+1)*4][::-1]:
            b1.append(t[0])
            b2.append(t[1])
    meas_b1 = np.array(b1, dtype=float); meas_b1 -= np.nanmin(meas_b1)
    meas_b2 = np.array(b2, dtype=float); meas_b2 -= np.nanmin(meas_b2)

    t0s = t_0 / 1000.0
    t1s = t_1 / 1000.0

    # --- 2. Costo con SURROGATE (velocissimo) ---
    def cost_surrogate(params):
        xc, yc, r, a_sq = params

        # Penalità
        pen = 0.0
        if r > 1.0:              pen += 1000.0 * (r - 1.0)
        d = np.hypot(xc, yc)
        if d > 1.5:              pen += 1000.0 * (d - 1.5)
        if a_sq > 4.0:           pen += 100.0  * (a_sq - 4.0)

        err = 0.0
        lam, delta = 150.0, 1.5
        for x_imp, y_imp, meas in [(x_0, y_0, meas_b1), (x_1, y_1, meas_b2)]:
            sim = surrogate_predict(x_imp, y_imp, xc, yc, r, a_sq)
            mask = ~np.isnan(meas)
            if np.any(mask):
                s, m = sim[mask], meas[mask]
                diff = np.abs(s - m)
                huber = np.where(diff <= delta, 0.5*diff**2, delta*(diff - 0.5*delta))
                corr = pearsonr(s, m)[0] if (np.std(s) > 1e-6 and np.std(m) > 1e-6) else 0.0
                err += np.mean(huber) + lam*(1.0 - corr)
        return err + pen

    # --- 3. Costo con FMM vero (per affinamento finale) ---
    def cost_fmm(params):
        xc, yc, r, a_sq = params
        a_ratio = float(a_sq**2)
        sigma_11, sigma_12, sigma_22 = 1.0, 0.0, a_ratio

        pen = 0.0
        if r > 1.0:        pen += 1000.0 * (r - 1.0)
        if np.hypot(xc,yc) > 1.5: pen += 1000.0 * (np.hypot(xc,yc) - 1.5)

        k = 50.0
        dist = np.hypot(X_mesh_vel - xc, Y_mesh_vel - yc)
        speed_mat = EST_V_SLOW + 0.5*(1+np.tanh(k*(dist-r)))*(EST_V_BASE - EST_V_SLOW)

        err = 0.0
        lam, delta = 150.0, 1.5
        for x_imp, y_imp, t_imp, meas in [
            (x_0, y_0, t0s, meas_b1), (x_1, y_1, t1s, meas_b2)
        ]:
            sim = anisotropic_FMM_GPT(x_imp, y_imp, t_imp, X_mesh_vel, Y_mesh_vel,
                                       speed_mat, sigma_11, sigma_12, sigma_22) * 1000.0
            s_t = sim[idx_vel[:,0], idx_vel[:,1]]
            s_t = s_t - np.nanmin(s_t)
            mask = ~np.isnan(meas)
            if np.any(mask):
                s, m = s_t[mask], meas[mask]
                diff = np.abs(s - m)
                huber = np.where(diff<=delta, 0.5*diff**2, delta*(diff-0.5*delta))
                corr = pearsonr(s, m)[0] if (np.std(s)>1e-6 and np.std(m)>1e-6) else 0.0
                err += np.mean(huber) + lam*(1.0 - corr)
        return err + pen

    # --- 4. Ricerca globale con surrogate (LHS 16384 punti in parallelo) ---
    from scipy.stats import qmc

    sampler = qmc.Sobol(d=4, seed=42)
    sample = sampler.random(n=16384)  # shape (16384, 4), valori in [0, 1]

    l_bounds = [-1.5, -1.5, 0.02, 1.0]
    u_bounds = [ 1.5,  1.5,  1.2, 5.0]
    random_starts = qmc.scale(sample, l_bounds, u_bounds)

    costs = Parallel(n_jobs=-1)(delayed(cost_surrogate)(p) for p in random_starts)
    best_p = random_starts[np.argmin(costs)]
    print(f'  Global best (surrogate): pos=({best_p[0]:.2f},{best_p[1]:.2f}), r={best_p[2]:.2f}, cost={min(costs):.2f}')

    # --- 5. Ottimizzazione locale con surrogate (Nelder-Mead, tolleranza stretta) ---
    bounds = [(-2.0, 2.0), (-2.0, 2.0), (0.01, 1.5), (1.0, 5.0)]
    res_surr = minimize(
        cost_surrogate, best_p, method='Nelder-Mead', bounds=bounds,
        options={'xatol': 0.005, 'fatol': 0.05, 'maxiter': 2000, 'adaptive': True}
    )

    # --- 6. Affinamento finale con FMM vero (poche iterazioni, parte già vicino al minimo) ---
    res_fmm = minimize(
        cost_fmm, res_surr.x, method='Nelder-Mead', bounds=bounds,
        options={'xatol': 0.01, 'fatol': 0.1, 'maxiter': 80, 'adaptive': True}
    )

    xc, yc, r, _ = res_fmm.x
    return [xc, yc, r]

## Test su tutti e 5 gli scenari

In [ ]:
# import time

# all_results = []
# print('=== TEST SU TUTTI I 5 SCENARI ===\n')

# for idx in range(5):
#     ev  = signals[idx, :, :]
#     x0, y0 = impulse_sites[idx][0]
#     x1, y1 = impulse_sites[idx][1]
#     t0_ = starting_times[idx][0]
#     t1_ = starting_times[idx][1]
#     tx, ty, tr = reference_targets[idx]

#     t_start = time.time()
#     print(f'--- Scenario {idx+1} ---')
#     pred = GroupSolutionCP2(ev, x0, y0, t0_, x1, y1, t1_)
#     elapsed = time.time() - t_start

#     px, py, pr = pred
#     ec = np.sqrt((px-tx)**2 + (py-ty)**2)
#     er = abs(pr - tr)
#     all_results.append(pred)

#     print(f'  Reale   : [{tx:7.3f}, {ty:7.3f}, {tr:7.3f}]')
#     print(f'  Stimato : [{px:7.3f}, {py:7.3f}, {pr:7.3f}]')
#     print(f'  Err ctr : {ec:.4f} | Err r: {er:.4f} | Tempo: {elapsed:.1f}s\n')

# print('\n=== RIEPILOGO ===')
# print(f'{"Sc":<4} {"x":<8} {"y":<8} {"r":<8} {"Err_c":<10} {"Err_r":<8}')
# for i, (p, ref) in enumerate(zip(all_results, reference_targets)):
#     ec = np.sqrt((p[0]-ref[0])**2 + (p[1]-ref[1])**2)
#     er = abs(p[2]-ref[2])
#     print(f'{i+1:<4} {p[0]:<8.3f} {p[1]:<8.3f} {p[2]:<8.3f} {ec:<10.4f} {er:<8.4f}')

=== TEST SU TUTTI I 5 SCENARI ===

--- Scenario 1 ---
  Global best (surrogate): pos=(-0.48,0.50), r=0.65, cost=19.02
  Reale   : [ -0.407,   0.299,   0.578]
  Stimato : [ -0.412,   0.556,   0.672]
  Err ctr : 0.2575 | Err r: 0.0940 | Tempo: 76.7s

--- Scenario 2 ---
  Global best (surrogate): pos=(1.03,-0.22), r=0.77, cost=7.04
  Reale   : [  0.810,   0.113,   0.394]
  Stimato : [  1.133,  -0.236,   0.802]
  Err ctr : 0.4762 | Err r: 0.4084 | Tempo: 65.0s

--- Scenario 3 ---
  Global best (surrogate): pos=(-1.09,-0.56), r=0.68, cost=12.90
  Reale   : [ -0.816,  -0.686,   0.274]
  Stimato : [ -1.203,  -0.486,   0.877]
  Err ctr : 0.4352 | Err r: 0.6027 | Tempo: 112.7s

--- Scenario 4 ---
  Global best (surrogate): pos=(0.18,-0.48), r=0.19, cost=4.99
  Reale   : [  0.197,  -0.242,   0.042]
  Stimato : [  0.149,  -0.230,   0.278]
  Err ctr : 0.0492 | Err r: 0.2363 | Tempo: 79.6s

--- Scenario 5 ---
  Global best (surrogate): pos=(0.75,-0.67), r=0.41, cost=6.91
  Reale   : [  0.647,  -0.6